# Week 5 Lab – Tidying, Cleaning, Imputation and Outliers

Four preprocessing workflows, one per dataset. All logic lives in one worker class per dataset under `src/`; each notebook cell calls one method.

| Section | Dataset | Focus |
|---|---|---|
| 1 | PEW Research Center (Spring 2025) | Tidying with `melt`, labelling codes from the survey metadata |
| 2 | Billboard Top 100 | Tidying with `melt` and string cleaning of weekly rankings |
| 3 | Cars | Data cleaning: irrelevant rows, types, duplicates, dropping vs imputing |
| 4 | Diabetes (Pima) | Outlier detection: box plot, scatter plot, Z-score, IQR |

Setup instructions are in `README.md`; design decisions and data sources are in `docs/PROJECT_DETAILS.md`.

## Replicability Notes

- **Environment:** every package is pinned in `requirements.txt`, created with `pip freeze` from the project virtual environment.
- **Different-machine test:** the notebook was cloned from GitHub and run end to end on a second Windows machine, in a new virtual environment (Python 3.12.7) built only from `requirements.txt`. All cells ran without errors.
- **Database:** Sections 3 and 4 read from and write to the team's Neon PostgreSQL database. The connection string goes in a local `.env` file (template: `.env.example`), which is not committed.
- **Run order:** run from the project root with **Restart Kernel → Run All**; later cells depend on earlier ones.

Issues found during testing and how they were resolved are listed in the [Replicability Check](#Replicability-Check) at the end.

# Section 1 – Tidying Data (PEW Research Dataset)

**Goal:** reshape the per-country income columns into a tidy long table, then use the survey's own codebook to turn numeric codes into readable labels.

**Data**
- `data/Pew Research Center Global Attitudes Spring 2025 Dataset CSV.csv`: 28,333 respondents in 24 countries, 483 columns.
- `data/Pew Research Center Global Attitudes Spring 2025 Metadata.xml`: the codebook (DDI format) with the meaning of every code.

Each country asked its own income question, so the data has one column per country (`d_income_canada`, `d_income_india`, ...) and a follow-up column (`d_income2_<country>`) for people who did not answer the first one. This breaks the tidy-data rule that each variable is one column.

## 1.1 Load the Dataset

In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px

from src.pew_worker import PewWorker

PROJECT_ROOT = Path.cwd()
print("Project root:", PROJECT_ROOT)

Project root: c:\Users\KING\Desktop\lab2_tidying_cleaning_imputation_outlier


In [2]:
pew_files = sorted((PROJECT_ROOT / "data").rglob("Pew*.csv"))

print("Pew files found:", len(pew_files))
for file in pew_files:
    print(file.name)

Pew files found: 1
Pew Research Center Global Attitudes Spring 2025 Dataset CSV.csv


In [3]:
pew = PewWorker(pew_files[0])
pew_df = pew.load()

print("Dataset shape:", pew_df.shape)
display(pew_df.head())

Dataset shape: (28333, 483)


,survey,country,weight,id,econ_sit,satisfied_democracy,polsys_satisfied,fav_us,fav_china,fav_eu,...,region_skorea,region_spain,region_sweden,region_turkey,region_uk,urbanicity,qdate_s,qdate_e,intdate_aus,phone_sample
0,1202501,2,0.592005,8001432,2,2.0,NaN,4,3,2,...,NaN,NaN,NaN,NaN,NaN,NaN,3/23/2025,3/23/2025,NaN,1.0
1,1202501,2,0.446814,8003472,4,3.0,NaN,4,4,2,...,NaN,NaN,NaN,NaN,NaN,NaN,3/11/2025,3/11/2025,NaN,1.0
2,1202501,2,1.120110,8006468,2,1.0,NaN,3,8,2,...,NaN,NaN,NaN,NaN,NaN,NaN,4/1/2025,4/1/2025,NaN,1.0
3,1202501,2,0.695317,8006970,2,1.0,NaN,4,8,1,...,NaN,NaN,NaN,NaN,NaN,NaN,3/10/2025,3/10/2025,NaN,1.0
4,1202501,2,0.369352,8008601,2,2.0,NaN,4,3,1,...,NaN,NaN,NaN,NaN,NaN,NaN,3/23/2025,3/23/2025,NaN,1.0


## 1.2 Reshape the Income Columns with `melt`

`melt_income()` keeps only the identifying columns (`id`, `country`, `weight`), stacks the 49 income columns into `income_variable` / `income_code`, drops questions the respondent was never asked, and splits the column name into `measure` (`income` or `income2`) and `country_name`.

In [4]:
pew_melted = pew.melt_income()

print("Reshaped dataset:", pew_melted.shape)
display(pew_melted.head())

pew_summary = pew.summary()
print("Summary shape:", pew_summary.shape)
display(pew_summary)

Reshaped dataset: (29438, 7)


,id,country,weight,income_variable,income_code,measure,country_name
0,2002686,23,0.356581,d_income_argentina,9,income,argentina
1,2002725,23,2.364549,d_income_argentina,4,income,argentina
2,2002731,23,1.512416,d_income_argentina,98,income,argentina
3,2002734,23,0.621898,d_income_argentina,7,income,argentina
4,2002918,23,0.472910,d_income_argentina,6,income,argentina


Summary columns: ['country_name', 'income', 'income2']


,country_name,income,income2
0,argentina,1001,99
1,brazil,1053,42
2,canada,1024,120
3,france,1006,52
4,germany,1006,71
5,greece,1010,28
6,hungary,1020,72
7,india,3545,198
8,indonesia,1022,3
9,israel,998,137


The result is one row per answered income question (29,438 rows, 7 columns) instead of 49 mostly empty columns. The UK appears as several rows because its question was split into annual, monthly and weekly versions. Australia is one of the 24 survey countries but has no income question in the public file, so 23 countries remain.

## 1.3 Add Labels from the Metadata

The income columns only store numeric codes: `5` is one bracket in Canada and a different one in India. `load_metadata()` reads the codebook and `add_metadata_columns()` adds three columns:

| Column | Content |
|---|---|
| `country_label` | Full country name for the numeric `country` code |
| `income_label` | The income bracket the respondent chose, in that country's currency |
| `response_status` | `valid`, `dont_know` or `refused` |

`response_status` is the most important one. Don't know and Refused are stored as numbers (98/99 in the main question, 8/9 in most follow-up questions), so without it they would be counted as income answers.

In [5]:
pew_xml = next((PROJECT_ROOT / "data").rglob("Pew*Metadata.xml"))

pew = PewWorker(pew_files[0], metadata_path=pew_xml)
pew_df = pew.load()
print("Metadata file:", pew_xml.name)
print("Variables with labels:", pew.load_metadata())

Metadata file: Pew Research Center Global Attitudes Spring 2025 Metadata.xml
Variables with labels: 483


In [6]:
pew_melted = pew.melt_income()
pew_melted = pew.add_metadata_columns()

print("Labelled dataset:", pew_melted.shape)
display(pew_melted[["country_label", "income_variable", "income_code",
                    "income_label", "response_status"]].sample(8, random_state=1))
print(pew_melted["response_status"].value_counts())

Enriched shape: (29438, 10)


,country_label,income_variable,income_code,income_label,response_status
10006,India,d_income_india,99,Refused (DO NOT READ),refused
21869,South Africa,d_income_safrica,9,"R4,501 to R8,000",valid
24748,Spain,d_income_spain,8,"2,501 to 3,000 Euros",valid
27835,UK,d_income_uk_cat,1,Annual,valid
25019,Spain,d_income_spain,4,"1,251 to 1,500 Euros",valid
25072,Spain,d_income_spain,12,"More than 4,500 Euros",valid
3237,Canada,d_income2_canada,9,Refused (DO NOT READ),refused
23537,South Korea,d_income_skorea,11,"6,000,001 to 8,000,000 Won",valid


response_status
valid        26353
refused       2003
dont_know     1082
Name: count, dtype: int64


## 1.4 Response Status by Country

Only the main income question (`measure == "income"`) is used, because the follow-up question is asked only to people who did not answer the first one. The UK item `d_income_uk_cat` is excluded: it asks whether the respondent prefers to answer annually, monthly or weekly, so it is not an income.

In [7]:
main_income = pew_melted[
    (pew_melted["measure"] == "income")
    & (pew_melted["income_variable"] != "d_income_uk_cat")
]

status_pct = (
    pd.crosstab(main_income["country_label"], main_income["response_status"],
                normalize="index")
    .mul(100).round(1)
    .sort_values("valid")
)
display(status_pct)

response_status,dont_know,refused,valid
country_label,,,
Japan,5.4,11.2,83.4
Italy,5.0,10.0,85.0
Poland,2.5,11.3,86.2
Israel,7.1,6.6,86.3
Canada,2.8,8.9,88.3
Mexico,5.8,5.0,89.2
Argentina,5.0,4.9,90.1
South Korea,5.2,3.9,90.9
Netherlands,3.1,5.4,91.5


In [8]:
status_long = status_pct.reset_index().melt(
    id_vars="country_label", var_name="response_status", value_name="percent"
)

fig = px.bar(
    status_long,
    x="percent", y="country_label", color="response_status",
    orientation="h",
    category_orders={"country_label": status_pct.index.tolist(),
                     "response_status": ["valid", "dont_know", "refused"]},
    color_discrete_map={"valid": "#2a9d8f", "dont_know": "#e9c46a", "refused": "#e76f51"},
    labels={"percent": "Share of respondents (%)", "country_label": "",
            "response_status": "Response"},
    title="Income question: valid answers vs Don't know / Refused",
)
fig.update_layout(barmode="stack", height=650, xaxis_range=[0, 100])
fig.show()

**Observations**
- Across all income questions, 26,353 answers are valid, 2,003 are Refused and 1,082 are Don't know.
- Japan has the lowest share of valid answers (83.4%, with 11.2% refusing), followed by Italy (85.0%) and Poland (86.2%).
- Indonesia (99.7%), Sweden (98.0%) and Kenya (97.8%) have almost no non-responses.
- Refusing is more common than Don't know in 15 of the 23 countries, which suggests income is treated as a private topic more often than an unknown one.

## 1.5 Valid Responses after Labelling

With `response_status` in place, `summary()` counts only valid answers. Comparing it with `pew_summary` from 1.2 shows how many non-responses were being counted as incomes.

In [9]:
pew_summary_valid = pew.summary()

comparison = pew_summary.merge(
    pew_summary_valid, on="country_name", suffixes=("_all", "_valid")
)
comparison["removed"] = (
    comparison["income_all"] + comparison["income2_all"]
    - comparison["income_valid"] - comparison["income2_valid"]
)

display(comparison)
print("Non-responses removed in total:", comparison["removed"].sum())

,country_name,income_all,income2_all,income_valid,income2_valid,removed
0,argentina,1001,99,902,60,138
1,brazil,1053,42,1011,18,66
2,canada,1024,120,904,56,184
3,france,1006,52,954,17,87
4,germany,1006,71,935,32,110
5,greece,1010,28,982,11,45
6,hungary,1020,72,948,32,112
7,india,3545,198,3347,89,307
8,indonesia,1022,3,1019,1,5
9,israel,998,137,861,62,212


Non-responses removed in total: 3085


**Observations**
- 3,085 non-responses were removed. Before labelling they would have been read as the highest income brackets (codes 98/99) or as extra categories (8/9).
- The follow-up `income2` question is affected the most, because it is only asked to people who already said Don't know or Refused: Argentina drops from 99 to 60 valid answers and Canada from 120 to 56.
- Without the metadata the 8/9 codes in `income2` look like ordinary answers, which is why the codebook, not a fixed list of codes, decides what counts as a non-response.

# Section 2 – Tidying Data (Billboard Dataset)

**Goal:** turn the Billboard Top 100 chart, stored with one column per week (`1st week` ... `65th week`), into a tidy table with one row per song per week, then plot ranking trajectories.

**Data:** `data/billboard.csv`, 317 songs that entered the chart in 1999–2000, 77 columns.

All the logic lives in `src/billboard_worker.py`.

## 2.1 Load the Raw Data

In [10]:
from src.billboard_worker import BillboardWorker

billboard_file = PROJECT_ROOT / "data" / "billboard.csv"
print("Billboard file found:", billboard_file.is_file())

Billboard file: c:\Users\KING\Desktop\lab2_tidying_cleaning_imputation_outlier\data\billboard.csv | exists: True


In [11]:
billboard = BillboardWorker(billboard_file)
billboard_df = billboard.load()

print("Raw shape:", billboard_df.shape)
display(billboard_df.head())

Raw shape: (317, 77)


,artist,track,duration,genre,date entered,date peaked,weeks to peak,weeks on chart,worst position,best position,...,56th week,57th week,58th week,59th week,60th week,61st week,62nd week,63rd week,64th week,65th week
0,Destiny's Child,Independent Women Part I,218,Rock,9/23/2000,11/18/2000,8,28,78,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Santana,"Maria, Maria",258,Rock,2/12/2000,4/8/2000,8,26,48,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Savage Garden,I Knew I Loved You,247,Rock,10/23/1999,1/29/2000,14,33,71,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Madonna,Music,225,Rock,8/12/2000,9/16/2000,5,24,44,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"Aguilera, Christina",Come On Over Baby (All I Want Is You),218,Rock,8/5/2000,10/14/2000,10,21,57,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2.2 Melt the Weekly Columns

The 65 week columns are values of one variable (the week), not separate variables. `melt_weeks()` stacks them into `week` and `rank`, keeping the song details as identifier columns.

In [12]:
bb_melted = billboard.melt_weeks()

print("Melted shape:", bb_melted.shape)
display(bb_melted[["artist", "track", "week", "rank"]].head())

Melted shape: (20605, 14)


,artist,track,week,rank
0,Destiny's Child,Independent Women Part I,1st week,78.0
1,Santana,"Maria, Maria",1st week,15.0
2,Savage Garden,I Knew I Loved You,1st week,71.0
3,Madonna,Music,1st week,41.0
4,"Aguilera, Christina",Come On Over Baby (All I Want Is You),1st week,57.0


## 2.3 Clean the Week Labels

Week labels such as `1st week` are text. `clean_week_column()` strips the ordinal suffix and converts them to integers so weeks can be sorted and used in calculations.

In [13]:
bb_melted = billboard.clean_week_column()

print("Week dtype:", bb_melted["week"].dtype)
print("Week range:", bb_melted["week"].min(), "to", bb_melted["week"].max())

Week dtype: int64
Week range: 1 to 65


## 2.4 Drop Weeks without a Ranking

A song only has a rank while it is on the chart, so most melted rows are empty.

In [14]:
before = len(bb_melted)
bb_tidy = billboard.drop_empty_ranks()

print(f"Rows: {before:,} → {len(bb_tidy):,} ({before - len(bb_tidy):,} empty weeks removed)")

Rows: 20,605 → 5,307 (15,298 empty weeks removed)


## 2.5 Add the Chart Date

`chart_date` = `date entered` + 7 days for every week after the first, giving each ranking a real calendar date.

In [15]:
bb_tidy = billboard.add_chart_date()

display(bb_tidy[["artist", "track", "date entered", "week", "chart_date", "rank"]].head(10))

,artist,track,date entered,week,chart_date,rank
0,Destiny's Child,Independent Women Part I,9/23/2000,1,2000-09-23,78.0
1,Santana,"Maria, Maria",2/12/2000,1,2000-02-12,15.0
2,Savage Garden,I Knew I Loved You,10/23/1999,1,1999-10-23,71.0
3,Madonna,Music,8/12/2000,1,2000-08-12,41.0
4,"Aguilera, Christina",Come On Over Baby (All I Want Is You),8/5/2000,1,2000-08-05,57.0
5,Janet,Doesn't Really Matter,6/17/2000,1,2000-06-17,59.0
6,Destiny's Child,Say My Name,12/25/1999,1,1999-12-25,83.0
7,"Iglesias, Enrique",Be With You,4/1/2000,1,2000-04-01,63.0
8,Sisqo,Incomplete,6/24/2000,1,2000-06-24,77.0
9,Lonestar,Amazed,6/5/1999,1,1999-06-05,81.0


## 2.6 Ranking Trajectories

Rank 1 is placed at the top of the y-axis.

In [16]:
billboard.plot_rankings().show()

**Observations**
- Melting 317 songs × 65 weeks gives 20,605 rows; 15,298 of them (74%) are weeks when the song was not on the chart and are removed, leaving 5,307 rankings.
- All three plotted songs reached number 1: *Music* (Madonna) after 5 weeks, *Maria, Maria* (Santana) after 8 and *Say My Name* (Destiny's Child) after 12. *Say My Name* climbed slowest but stayed longest (32 weeks).

# Section 3 – Data Cleaning (Cars Dataset)

**Goal:** prepare the Cars dataset for analysis by removing irrelevant rows, fixing data types, merging public records, removing duplicates and handling missing values by dropping or imputing.

**Data**
- `data/cars.csv`: 406 cars (1970–1982), semicolon-separated, stored in Neon as `cars_raw` (all values as text, as in the file).
- Public records: a fixed sample of 1,000 conventional petrol and diesel vehicles (1984–1989) from the US EPA fuel economy file `data/vehicles.csv.zip`, stored in Neon as `cars_public_epa`.

All the logic lives in `src/cars_worker.py`. The first cell uploads both sources to Neon; re-running replaces the rows, so the section always starts from the original data.

In [17]:
from src.cars_worker import CarsWorker

cars = CarsWorker()
display(cars.upload_sources())

## 3.1 Load and Remove Irrelevant Rows

The second line of `cars.csv` holds data-type labels (`STRING;DOUBLE;INT;...`), not a car. `drop_irrelevant_rows()` removes any row whose values are all type labels.

In [19]:
cars_df = cars.load()
print("Loaded:", cars_df.shape)

cars_df = cars.drop_irrelevant_rows()
print("After dropping the STRING;DOUBLE;... row:", cars_df.shape)
display(cars_df.head())

Loaded: (407, 9)
After dropping the STRING;DOUBLE;... row: (406, 9)


,Car,MPG,Cylinders,Displacement,Horsepower,Weight,Acceleration,Model,Origin
0,Chevrolet Chevelle Malibu,NaN,8,307.0,130.0,3504.,12.0,70,US
1,Buick Skylark 320,15.0,8,350.0,NaN,3693.,11.5,70,US
2,Plymouth Satellite,NaN,8,318.0,150.0,3436.,11.0,70,US
3,AMC Rebel SST,16.0,8,NaN,150.0,NaN,12.0,70,US
4,Ford Torino,17.0,8,302.0,140.0,3449.,10.5,70,US


## 3.2 Convert Data Types

`zero_report()` runs first, while the values are still text, because `convert_types()` turns impossible zeros into `NaN`. A car cannot have 0 MPG or 0 horsepower, so those zeros are missing measurements.

In [20]:
print("Zero values per column:")
display(cars.zero_report())

cars_df = cars.convert_types()
display(cars_df.dtypes)
display(cars.missing_report())

Zero values per column:


MPG             8
Cylinders       0
Displacement    0
Horsepower      6
Weight          0
Acceleration    0
Model           0
dtype: int64

Car                  str
MPG              float64
Cylinders          int64
Displacement     float64
Horsepower       float64
Weight           float64
Acceleration     float64
Model              int64
Origin          category
dtype: object

,missing,missing_%,valid
Car,0,0.00,406
MPG,11,2.71,395
Cylinders,0,0.00,406
Displacement,1,0.25,405
Horsepower,8,1.97,398
Weight,1,0.25,405
Acceleration,0,0.00,406
Model,0,0.00,406
Origin,0,0.00,406


**Observations:** `MPG` had 8 zeros and `Horsepower` 6. After conversion `MPG` has 11 missing values (2.71%) and `Horsepower` 8 (1.97%); `Displacement` and `Weight` have one each.

## 3.3 Add Public Records (US EPA)

The EPA sample is translated to the lab schema: make + model → `Car`, combined MPG → `MPG`, litres × 61.0237 → `Displacement` (cubic inches), model year → `Model`, and make → `Origin` (US, Europe, Japan). The EPA file does not publish horsepower, weight or acceleration, so these stay `NaN`. A `source` column keeps the two origins apart.

In [21]:
cars_df = cars.merge_public_data()
display(cars.source_summary())
display(cars.missing_report())

,rows,first_year,last_year
source,,,
epa_public,1000,84,89
original,406,70,82


,missing,missing_%,valid
Car,0,0.00,1406
MPG,11,0.78,1395
Cylinders,0,0.00,1406
Displacement,1,0.07,1405
Horsepower,1008,71.69,398
Weight,1001,71.19,405
Acceleration,1000,71.12,406
Model,0,0.00,1406
Origin,0,0.00,1406
source,0,0.00,1406


**Observations:** the combined data has 1,406 cars covering 1970–1989. About 71% of `Horsepower`, `Weight` and `Acceleration` are now missing, all from the EPA rows.

## 3.4 Remove Duplicates

`duplicates_report()` lists every row that has an exact copy; `drop_duplicates()` keeps the first occurrence.

In [22]:
dups = cars.duplicates_report()
print("Rows with an exact duplicate:", len(dups))
display(dups)

cars_df = cars.drop_duplicates()
print("After dropping duplicates:", cars_df.shape)

Rows with an exact duplicate: 94


,Car,MPG,Cylinders,Displacement,Horsepower,Weight,Acceleration,Model,Origin,source
410,Aston Martin Saloon/Vantage/Volante,8.0,8.0,323.4,NaN,NaN,NaN,85,Europe,epa_public
554,Aston Martin Saloon/Vantage/Volante,8.0,8.0,323.4,NaN,NaN,NaN,85,Europe,epa_public
770,BMW 6 Series,16.0,6.0,207.5,NaN,NaN,NaN,87,Europe,epa_public
771,BMW 6 Series,16.0,6.0,207.5,NaN,NaN,NaN,87,Europe,epa_public
798,Buick Somerset/Skylark,23.0,4.0,152.6,NaN,NaN,NaN,87,US,epa_public
...,...,...,...,...,...,...,...,...,...,...
861,Toyota Truck 2WD,21.0,4.0,146.5,NaN,NaN,NaN,87,Japan,epa_public
1129,Toyota Truck 2WD,20.0,4.0,146.5,NaN,NaN,NaN,89,Japan,epa_public
1131,Toyota Truck 2WD,20.0,4.0,146.5,NaN,NaN,NaN,89,Japan,epa_public
1338,Volkswagen Quantum,31.0,4.0,97.6,NaN,NaN,NaN,84,Europe,epa_public


After dropping duplicates: (1358, 10)


**Observations:** all 94 duplicated rows come from the EPA sample. The EPA lists separate entries per transmission and drive type, which become identical once only the lab columns are kept. Removing 48 copies leaves 1,358 cars.

## 3.5 Drop or Impute Missing Values

`drop_missing()` and `impute()` return copies, so the cost of each option can be checked without changing the data.

In [23]:
print("Drop rows with any NaN:   ", cars.drop_missing(axis=0).shape)
print("Drop columns with any NaN:", cars.drop_missing(axis=1).shape)

for strategy in ["mean", "median", "mode"]:
    filled = cars.impute("MPG", strategy=strategy)
    print(f"{strategy:>6} → MPG mean after fill: {filled['MPG'].mean():.2f}")

Drop rows with any NaN:    (387, 10)
Drop columns with any NaN: (1358, 5)
mean → MPG mean after fill: 20.73
median → MPG mean after fill: 20.72
mode → MPG mean after fill: 20.71


**Observations:** dropping rows would keep only 387 of 1,358 cars (all EPA rows lost), and dropping columns would keep only 5 of 10 columns. Both lose too much information, so the missing values are imputed. For `MPG` the choice of simple strategy barely matters (20.71–20.73) because only 11 values are missing.

## 3.6 Choose Mean or Median

For a skewed column the mean is pulled toward the long tail, so the median is a better fill value.

In [24]:
for col in ["MPG", "Displacement", "Horsepower", "Weight"]:
    print(col, cars.center_stats(col).to_dict())

cars.distribution_plot("Horsepower").show()
cars.distribution_plot("MPG").show()

MPG {'mean': 20.727, 'median': 20.0, 'skew': 0.861}
Displacement {'mean': 193.189, 'median': 152.6, 'skew': 0.755}
Horsepower {'mean': 104.698, 'median': 95.0, 'skew': 1.049}
Weight {'mean': 2978.294, 'median': 2815.0, 'skew': 0.509}


**Observations:** every column is right-skewed. `Horsepower` is the most skewed (1.05): its mean (104.7) is almost 10 above its median (95). The median is therefore used as the fallback strategy.

## 3.7 Impute Missing Values

A single median for about 950 EPA rows would stack identical values and distort the distribution, so related columns are used where possible:

| Column | Method | Reason |
|---|---|---|
| `MPG` | Median within each `Cylinders` group | Fuel economy depends strongly on engine size |
| `Weight` | Linear regression on `Displacement`, `Cylinders` | Larger engines go in heavier cars |
| `Horsepower` | Linear regression on `Displacement`, `Cylinders`, `Weight` | Uses the weight imputed in the previous step |
| Remaining (`Acceleration`, ...) | Median with scikit-learn `SimpleImputer` | No reliable predictor is available |

In [25]:
cars.group_impute("MPG", by="Cylinders", strategy="median")
cars.regression_impute("Weight", ["Displacement", "Cylinders"])
cars.regression_impute("Horsepower", ["Displacement", "Cylinders", "Weight"])
cars.simple_impute(["Acceleration", "Displacement", "Horsepower", "Weight"], strategy="median")

display(cars.missing_report())

,missing,missing_%,valid
Car,0,0.0,1358
MPG,0,0.0,1358
Cylinders,0,0.0,1358
Displacement,0,0.0,1358
Horsepower,0,0.0,1358
Weight,0,0.0,1358
Acceleration,0,0.0,1358
Model,0,0.0,1358
Origin,0,0.0,1358
source,0,0.0,1358


## 3.8 Compare Before and After Cleaning

"Before" is the merged data before duplicates were removed and values imputed. The cleaned data is then saved to the `cars_clean` table in Neon.

In [26]:
display(cars.compare_before_after())
cars.before_after_plot("Horsepower").show()

display(cars.save_clean())

,before,after
rows,1406.00,1358.00
missing_cells,3021.00,0.00
duplicate_rows,48.00,0.00
MPG_mean,20.65,20.69
MPG_median,20.00,20.00
MPG_std,6.23,6.27
Displacement_mean,194.26,193.16
Displacement_median,152.60,152.60
Displacement_std,93.10,92.99
Horsepower_mean,104.70,104.27


,table,rows_written
0,cars_clean,1358


**Observations**
- The data went from 1,406 rows with 3,021 missing cells and 48 duplicates to 1,358 rows with none.
- `MPG` is almost unchanged (mean 20.65 → 20.69, median 20.0).
- The `Weight` and `Horsepower` medians drop (2,815 → 2,669 and 95 → 93) because the added EPA cars from the mid-1980s are smaller and lighter.
- Their standard deviations shrink (`Weight` 848 → 726, `Horsepower` 38.5 → 31.7): imputed values lie on the regression line, so they add less spread than real measurements would. `Acceleration` for the EPA rows is a single median value, which is a known limitation.

## 3.9 Key Takeaways

- Look for invalid values (zeros, label rows) before counting missing data, or they are treated as real measurements.
- Dropping rows or columns is only acceptable when little data is lost; here it would have removed 71% of the cars.
- Imputation from related columns preserves relationships better than a single fill value, but it still reduces variance and should be documented.

# Section 4 – Outlier Detection (Diabetes Dataset)

**Goal:** find observations that differ strongly from the rest of the data with visual methods (box plot, scatter plot) and statistical rules (Z-score, IQR), compare what each method flags, and remove outliers only where there is a reason to.

**Data**
- `data/diabetes.csv`: Pima Indians Diabetes dataset (768 women aged 21+, 9 columns), stored in Neon as `diabetes_raw`.
- Public records: 1,000 women aged 21+ from the **NHANES 2009–2012** survey of the US CDC (`data/nhanes_2009_2012_raw.csv.zip`, raw records published with the NHANES R package), stored in Neon as `diabetes_public_nhanes`. They are translated to the Pima columns before the analysis.

All the logic lives in `src/diabetes_worker.py`. The first cell uploads both sources to Neon; re-running replaces the rows, so the section always starts from the original data.

In [27]:
from src.diabetes_worker import DiabetesWorker

diabetes = DiabetesWorker()
display(diabetes.upload_sources())

,rows_uploaded
diabetes_raw,768
diabetes_public_nhanes,1000


## 4.1 Load and Explore the Diabetes Dataset

The data is read back from Neon (`diabetes_raw`) so every team member works on the same copy.

In [28]:
diabetes_df = diabetes.load()
print("Shape:", diabetes_df.shape)
print("Columns:", diabetes_df.columns.tolist())
display(diabetes_df.dtypes)
display(diabetes_df.head())
display(diabetes_df.describe().round(2))

Shape: (768, 9)
Columns: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']


Pregnancies                   int64
Glucose                       int64
BloodPressure                 int64
SkinThickness                 int64
Insulin                       int64
BMI                         float64
DiabetesPedigreeFunction    float64
Age                           int64
Outcome                       int64
dtype: object

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.00,768.00,768.00,768.00,768.00,768.00,768.00,768.00,768.00
mean,3.85,120.89,69.11,20.54,79.80,31.99,0.47,33.24,0.35
std,3.37,31.97,19.36,15.95,115.24,7.88,0.33,11.76,0.48
min,0.00,0.00,0.00,0.00,0.00,0.00,0.08,21.00,0.00
25%,1.00,99.00,62.00,0.00,0.00,27.30,0.24,24.00,0.00
50%,3.00,117.00,72.00,23.00,30.50,32.00,0.37,29.00,0.00
75%,6.00,140.25,80.00,32.00,127.25,36.60,0.63,41.00,1.00
max,17.00,199.00,122.00,99.00,846.00,67.10,2.42,81.00,1.00


### Invalid Values Before Outlier Analysis

There are no `NaN` values in the file, but `describe()` shows a minimum of `0` for `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin` and `BMI`. A living person cannot have a BMI or a diastolic blood pressure of 0, so these zeros are unrecorded measurements, not real values. `Pregnancies = 0` is valid and is kept.

In [29]:
display(diabetes.zero_report())

,zeros,zeros_%
Pregnancies,111,14.45
Glucose,5,0.65
BloodPressure,35,4.56
SkinThickness,227,29.56
Insulin,374,48.70
BMI,11,1.43
DiabetesPedigreeFunction,0,0.00
Age,0,0.00
Outcome,500,65.10


The zeros are converted to `NaN` so they are ignored by the outlier methods. Left as numbers, they would be flagged as extreme low values and distort the mean and standard deviation. They are **not** deleted: the rows stay in the data.

In [30]:
diabetes_df = diabetes.zeros_to_nan()
display(diabetes.missing_report())

,missing,missing_%,valid
Pregnancies,0,0.00,768
Glucose,5,0.65,763
BloodPressure,35,4.56,733
SkinThickness,227,29.56,541
Insulin,374,48.70,394
BMI,11,1.43,757
DiabetesPedigreeFunction,0,0.00,768
Age,0,0.00,768
Outcome,0,0.00,768


### Adding the Public Records (NHANES)

The NHANES women are translated to the Pima columns:

| Pima column | NHANES source |
|---|---|
| `Pregnancies` | `nPregnancies` (number of times pregnant) |
| `BloodPressure` | `BPDiaAve` (average diastolic pressure, mm Hg, the same measure Pima uses) |
| `BMI` | `BMI` |
| `Age` | `Age` (NHANES top-codes ages above 80 as 80) |
| `Outcome` | `Diabetes` ("told by a doctor they have diabetes": Yes = 1, No = 0) |
| `Glucose`, `SkinThickness`, `Insulin`, `DiabetesPedigreeFunction` | not collected in this NHANES file, so left as `NaN` |

A `source` column keeps the two origins apart.

In [31]:
diabetes_df = diabetes.merge_public_data()
display(diabetes.source_summary())
display(diabetes.missing_report())

,rows,mean_age,mean_bmi,diabetes_rate
source,,,,
nhanes_public,1000,50.235,29.344,0.147
pima_original,768,33.241,32.457,0.349


,missing,missing_%,valid
Pregnancies,242,13.69,1526
Glucose,1005,56.84,763
BloodPressure,42,2.38,1726
SkinThickness,1227,69.40,541
Insulin,1374,77.71,394
BMI,11,0.62,1757
DiabetesPedigreeFunction,1000,56.56,768
Age,0,0.00,1768
Outcome,0,0.00,1768
source,0,0.00,1768


The NHANES women are older on average (about 50 vs 33 years) and have a lower diabetes rate (14.7% vs 34.9%), so the combined data covers a wider population. The outlier analysis focuses on the features that both sources share: `BloodPressure`, `BMI`, `Pregnancies` and `Age`. `Glucose`, `Insulin` and `SkinThickness` are analysed on the Pima rows only.

## 4.2 Box Plot Method

In a box plot the **box** spans the first quartile (Q1, 25%) to the third quartile (Q3, 75%), and the line inside is the **median**. The **whiskers** reach the most extreme values that are still within 1.5 × IQR of the box. Points drawn beyond the whiskers are potential outliers.

`Insulin` is plotted separately because its scale (up to 846) would flatten the other box plots.

In [32]:
diabetes.box_plot(["Glucose", "BloodPressure", "BMI"]).show()
diabetes.box_plot("Insulin").show()

**Observations**
- `BloodPressure` has points beyond **both** whiskers: very low diastolic readings (below 38 mm Hg) and some high ones (above 102 mm Hg).
- `BMI` only has points above the upper whisker (around 49 to 68).
- `Glucose` has no points beyond the whiskers once the invalid zeros are removed.
- `Insulin` has a long upper tail, with many points above the upper whisker: it is the most skewed feature.

## 4.3 Scatter Plot Method

A scatter plot of two features shows points that sit apart from the main cloud, even when each value alone looks normal.

In [33]:
diabetes.scatter_plot("BMI", "BloodPressure").show()

**Observations:** most women fall in a cloud between roughly BMI 20–45 and diastolic pressure 50–95 mm Hg, and both sources overlap. The isolated points are at the bottom (diastolic pressure below 38 mm Hg; 14 of these 17 readings come from NHANES) and at the far right (BMI above 55).

## 4.4 Z-Score Method

$$z = \frac{x - \mu}{\sigma}$$

A value is flagged when $|z| > 3$, that is, more than 3 standard deviations from the mean.

In [34]:
z_bp = diabetes.zscore_outliers("BloodPressure", threshold=3)
print("BloodPressure outliers (|z| > 3):", len(z_bp))
display(z_bp[["BloodPressure", "z_score", "source"]].sort_values("BloodPressure"))

z_bmi = diabetes.zscore_outliers("BMI", threshold=3)
print("BMI outliers (|z| > 3):", len(z_bmi))

BloodPressure outliers (|z| > 3): 22


,BloodPressure,z_score,source
1088,18.0,-4.22,nhanes_public
1066,19.0,-4.14,nhanes_public
955,23.0,-3.82,nhanes_public
597,24.0,-3.74,pima_original
1221,28.0,-3.41,nhanes_public
778,29.0,-3.33,nhanes_public
769,30.0,-3.25,nhanes_public
18,30.0,-3.25,pima_original
1751,30.0,-3.25,nhanes_public
125,30.0,-3.25,pima_original


BMI outliers (|z| > 3): 17


## 4.5 Interquartile Range (IQR) Method

$$IQR = Q3 - Q1$$

$$Lower\ Bound = Q1 - 1.5 \times IQR \qquad Upper\ Bound = Q3 + 1.5 \times IQR$$

In [35]:
for col in ["BloodPressure", "BMI"]:
    print(col, diabetes.iqr_bounds(col).to_dict())
    print("  IQR outliers:", len(diabetes.iqr_outliers(col)))

BloodPressure {'Q1': 62.0, 'Q3': 78.0, 'IQR': 16.0, 'lower': 38.0, 'upper': 102.0}
  IQR outliers: 32
BMI {'Q1': 25.3, 'Q3': 34.8, 'IQR': 9.5, 'lower': 11.05, 'upper': 49.05}
  IQR outliers: 29


## 4.6 Compare Detection Methods

The box plot whiskers use the same 1.5 × IQR rule, so the points beyond the whiskers in 4.2 are exactly the `iqr_outliers` counted here.

In [36]:
features = ["Glucose", "BloodPressure", "BMI", "Insulin", "SkinThickness", "Pregnancies", "Age"]
display(diabetes.compare_methods(features))

,valid_values,zscore_outliers,iqr_outliers,flagged_by_both,skewness
feature,,,,,
Glucose,763,0,0,0,0.53
BloodPressure,1726,22,32,22,-0.09
BMI,1757,17,29,17,0.87
Insulin,394,8,24,8,2.17
SkinThickness,541,2,3,2,0.69
Pregnancies,1526,28,70,28,1.22
Age,1768,0,0,0,0.56


**Why the methods disagree**
- The IQR rule flags **more** values in every feature (158 in total vs 77 for the Z-score), and every Z-score outlier is also an IQR outlier here.
- The Z-score uses the mean and standard deviation, which are pulled by the extreme values themselves, and it assumes a roughly normal distribution. It works best on symmetric features such as `BloodPressure` (skewness −0.09).
- The IQR uses quartiles, which extreme values do not move, so it is more robust for skewed features. The gap is largest for `Insulin` (skewness 2.17: 8 vs 24) and `Pregnancies` (skewness 1.22: 28 vs 70).

## 4.7 Treat Outliers Carefully

Detecting an outlier does not mean it should be deleted. The decision per feature:

| Feature | Decision | Reason |
|---|---|---|
| `BloodPressure` **below** the lower fence (< 38 mm Hg) | **Remove** (17 rows) | A diastolic pressure of 18–37 mm Hg in a woman who was able to attend the exam is not plausible: these are measurement or recording errors |
| `BloodPressure` above the upper fence (> 102 mm Hg) | Keep | Severe hypertension is rare but real, and relevant to diabetes |
| `BMI` 49–68 | Keep | Severe obesity is a real condition and a known diabetes risk factor |
| `Insulin`, `Pregnancies` | Keep | Long right tails of genuine values (e.g. 10+ pregnancies) |

The cleaned data is a **copy**; the original `diabetes_df` is not overwritten.

In [37]:
diabetes_clean = diabetes.remove_outliers("BloodPressure", method="iqr", side="lower")
print("Rows before:", len(diabetes_df), "| rows after:", len(diabetes_clean))
display(diabetes_clean["source"].value_counts())
display(diabetes.compare_before_after(["BloodPressure", "BMI", "Glucose"]))

Rows before: 1768 | rows after: 1751


source
nhanes_public    986
pima_original    765
Name: count, dtype: int64

,before,after
rows,1768.00,1751.00
BloodPressure_mean,70.09,70.49
BloodPressure_median,70.00,70.00
BloodPressure_std,12.33,11.69
BMI_mean,30.69,30.66
BMI_median,29.79,29.75
BMI_std,7.41,7.39
Glucose_mean,121.69,121.80
Glucose_median,117.00,117.00
Glucose_std,30.54,30.54


In [38]:
diabetes.before_after_box("BloodPressure").show()
display(diabetes.save_clean())

,table,rows_written
0,diabetes_clean,1751


Removing the 17 implausible readings raises the `BloodPressure` mean from 70.09 to 70.49 mm Hg and lowers its standard deviation from 12.33 to 11.69. The median stays at 70, which shows how much less the median reacts to extreme values than the mean. `BMI` and `Glucose` barely change, and the diabetes rate stays at 23.5%, so the treatment does not bias the outcome. The cleaned copy is saved to the `diabetes_clean` table in Neon.

## 4.8 Diabetes Reflection

**1. What is an outlier?**
An observation that differs substantially from most of the data, such as a diastolic pressure of 18 mm Hg when the middle half of the women are between 62 and 78.

**2. What are at least three possible causes of outliers?**
Measurement errors (a faulty cuff reading), data-entry or recording errors (a 0 typed for a missing value), and genuine but rare cases (severe obesity or hypertension). Mixing populations can also create them: here the NHANES women are older than the Pima women.

**3. Which variables showed the clearest visual outliers?**
`BloodPressure`, on both sides, and `Insulin`, with a long upper tail. `BMI` has clear high outliers. `Glucose` has none once the invalid zeros are removed.

**4. How many observations were flagged by the Z-Score method?**
With |z| > 3: 22 for `BloodPressure` and 17 for `BMI`; 77 across the seven features compared in 4.6.

**5. How many were flagged by the IQR method?**
32 for `BloodPressure` and 29 for `BMI`; 158 across the seven features.

**6. Why can Z-Score and IQR produce different results?**
The Z-score depends on the mean and standard deviation, which outliers inflate, and it assumes a normal distribution. The IQR depends on quartiles, which outliers do not move. On skewed features (`Insulin`, `Pregnancies`) the IQR flags many more values.

**7. When should an outlier be retained?**
When it is a plausible, real observation, such as a BMI of 60 or a diastolic pressure of 110, especially when it carries information about the outcome being studied.

**8. When might removing an outlier be justified?**
When there is evidence it is an error: a value that is physically impossible or implausible (a diastolic pressure of 18 mm Hg), a data-entry mistake, or a record outside the population being studied.

**9. How did removal affect the descriptive statistics?**
Removing 17 rows (1,768 → 1,751) moved the `BloodPressure` mean from 70.09 to 70.49 and its standard deviation from 12.33 to 11.69, while the median stayed at 70. The other features and the diabetes rate barely changed.

**10. How could untreated outliers affect a downstream ML model?**
They can pull regression coefficients and the mean and standard deviation used for scaling, dominate distance-based models such as KNN, and make the model fit errors instead of patterns. Removing genuine extreme cases, on the other hand, would hide exactly the high-risk patients the model should learn about.

### Key takeaways
- Check for invalid values (zeros used as "missing") before looking for outliers, or the methods flag the wrong things.
- Box plots and the IQR rule are the same rule seen two ways; the Z-score is stricter and assumes normality, the IQR is more robust for skewed data.
- An outlier is a reason to investigate, not to delete: only the implausible blood-pressure readings were removed, on a copy of the data.

## Replicability Check

The notebook was run end to end in a new virtual environment on a second machine, set up only from the repository and `requirements.txt`. The issues below were found and resolved.

| Issue | Cause | Resolution |
|---|---|---|
| `ValueError: No income columns were found` in Section 1 | `rglob("*.csv")` picked up `billboard.csv` after more CSV files were added to `data/` | Search for `Pew*.csv` only |
| Changes to `src/*.py` were not picked up | Jupyter keeps imported modules in memory | **Restart Kernel**, then **Run All** |
| `RuntimeError: DATABASE_URL is not set` on a fresh clone | `.env` is excluded from Git so credentials are not published | Copy `.env.example` to `.env` and paste the connection string shared privately with the team |
| `FileNotFoundError` for files in `data/` | The workers use paths relative to the project root | Open and run the notebook from the project root folder |